In [2]:
from typing import TypedDict

class Car(TypedDict):
    make: str
    model: str
    year: int
    electric: bool

In [3]:
c=Car(make="Tesla",model="Model 3",year=2020,electric=True)
c1=Car(make="Ford",model="Mustang",year="1967",electric=False) #passing the  values into str inplace of int  for year field b ut this won't create any issue as this is not validating on runtime this is making hint for developer only
print(c)
print(c1)

{'make': 'Tesla', 'model': 'Model 3', 'year': 2020, 'electric': True}
{'make': 'Ford', 'model': 'Mustang', 'year': '1967', 'electric': False}


In [4]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
import os
from dotenv import load_dotenv
load_dotenv()

def model():
    model =ChatHuggingFace(llm=HuggingFaceEndpoint(
        repo_id="openai/gpt-oss-20b",
        task="text-generation",
        temperature=0,
        max_new_tokens=1024,
        huggingfacehub_api_token=os.getenv("HUGGINFACE_API_KEY"))
    )
    return model
model=model()

## How TypedDict guides LLM output
When you use a `TypedDict` as a schema for structured output with an LLM, a prompt is generated behind the scenes to instruct the model. For example:
```
You are an AI assistant that extracts structured insights from a given review. Extract:
- summary: a brief summary of the review
- sentiment: overall tone of the review (positive, neutral, negative)
Return the result as JSON.
```
This ensures the model's response matches the structure defined by the `TypedDict`.

In [5]:
from typing import TypedDict,Annotated,Optional
#schema 
class review(TypedDict):
    summary: Annotated[str, "A concise summary of the review"]
    sentiment: Annotated[str, "The overall sentiment of the review, e.g., positive, negative, neutral"]

structed_model=model.with_structured_output(review)

result=structed_model.invoke("Overall, my experience has been quite good. The platform is easy to use, the features work as expected, and the overall performance is smooth. Customer support could be a bit faster, but the service itself delivers what it promises. With a few improvements, this could easily become an excellent experience. Would recommend it to others.")
print(result)
print(result['summary'])
print(result['sentiment'])  

{'sentiment': 'positive', 'summary': 'Overall positive experience with smooth performance and useful features, but customer support could be faster. Recommend with minor improvements.'}
Overall positive experience with smooth performance and useful features, but customer support could be faster. Recommend with minor improvements.
positive


In [6]:
# we are adding big review and we are also adding 
from typing import TypedDict,Annotated,Optional
# #schema 
class review(TypedDict):
    key_thems=Annotated[list[str],"Key themes discussed in the review"]
    summary: Annotated[str, "A concise summary of the review"]
    sentiment: Annotated[str, "The overall sentiment of the review, e.g., positive, negative, neutral"]
    pros: Annotated[Optional[list[str]], "List of pros mentioned in the review"]
    cons: Annotated[Optional[list[str]],"LIST OF CONS MENTIONS IN THE REVIEW"]
    name:Annotated[Optional[str],"name of reviewer"]

structed_model=model.with_structured_output(review)

result=structed_model.invoke("""I’ve been using the iPhone 14 for a good amount of time now, and overall, it delivers a very polished and reliable smartphone experience—exactly what people expect from Apple.

The design is familiar but refined. Apple hasn’t changed the overall look drastically, but the phone feels premium, well-balanced, and solid in hand. The glass back and aluminum frame give it a sturdy feel, and the phone doesn’t feel bulky despite its premium build. It’s comfortable to use one-handed and fits easily into daily life.

The display is one of the highlights. The Super Retina XDR display is sharp, bright, and color-accurate. Whether you’re watching videos, scrolling through social media, or reading for long periods, the screen is easy on the eyes. Outdoor visibility is excellent, and the overall viewing experience feels smooth and premium, even though the refresh rate remains standard.

In terms of performance, the iPhone 14 is extremely reliable. Everyday tasks like browsing, multitasking, camera usage, and app switching feel effortless. Apps open quickly, animations are smooth, and the phone rarely slows down. Even with heavy usage, the device remains stable and efficient, which makes it feel dependable over the long term.

The camera system performs very well, especially for everyday photography. Photos come out sharp with accurate colors and strong dynamic range. Night photography is noticeably improved, producing clearer and brighter images without excessive noise. Video recording is a major strength—stabilization, clarity, and color consistency are among the best in the smartphone market. The front camera also performs well for selfies and video calls.

Battery life is decent and consistent. The phone comfortably lasts a full day with moderate to heavy usage, including browsing, media consumption, and navigation. While it may not be the longest-lasting battery in its class, Apple’s optimization ensures predictable and stable performance throughout the day.

The software experience (iOS) is one of the iPhone 14’s strongest points. The interface is clean, intuitive, and well-optimized. iOS feels smooth and polished, with long-term software updates being a major advantage. Security, privacy features, and ecosystem integration—especially with other Apple devices like MacBooks, iPads, and AirPods—add significant value.

One notable aspect is the ecosystem experience. Features like AirDrop, iMessage, FaceTime, and seamless device syncing make daily tasks easier, especially if you’re already using Apple products. This interconnected experience is something many users appreciate over time.

However, the iPhone 14 does have some limitations. Charging speed is relatively slow compared to competitors, and the absence of major design changes may feel underwhelming for users upgrading from recent iPhone models. Additionally, the lack of expandable storage and limited customization options may not suit everyone.

Build quality and reliability are excellent. The phone feels durable, and features like water resistance and improved safety options add an extra layer of confidence for daily use.
""")
print(result)
print("pros:",result['pros'],end="\n")
print("cons:",result['cons'],end="\n")
#print("name:",result['name'],end="\n")


{'cons': ['Slow charging speed compared to competitors', 'No major design changes, which may feel underwhelming', 'No expandable storage option', 'Limited customization options', 'Battery not the longest-lasting in its class'], 'name': '', 'pros': ['Polished design and premium feel', 'Super Retina XDR display with excellent brightness, color accuracy and outdoor visibility', 'Extremely reliable performance with smooth multitasking and app launches', 'Excellent camera performance, especially for video recording and night photography', 'Consistent battery life that lasts a full day', 'Intuitive iOS interface with long-term updates, strong security and privacy', 'Seamless ecosystem integration with other Apple devices', 'High build quality, durability and water resistance'], 'sentiment': 'positive', 'summary': 'The iPhone\u202f14 offers a premium, reliable experience with a polished design, sharp display, smooth performance, strong camera and video capabilities, solid battery life, and a 

# Pydantic 
1. Pydantic is a Python library that validates and parses data using type hints.
You define what data should look like, and Pydantic makes sure reality obeys.

Think of it as:

“Type hints that actually do something at runtime.”
2. The problem Pydantic solves

In plain Python, this is legal:

```python
age = "twenty five"
```

Python shrugs. Your app breaks later.

Pydantic steps in and says:

“No. If you said age: int, then it better be an integer—or I’ll stop you immediately.”

```python
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int
    email: str
```
``` python
user = User(name="Vinod", age="25", email="a@b.com")
# works >> "25"converted to 25
User(name="Vinod", age="abc", email="a@b.com")
Error: >> invalid integer
```
One-line takeaway

TypedDict = documentation + static hints

Pydantic = enforced data contracts at runtime

If data comes from outside your code (API, user, file) → Pydantic
If data stays inside trusted code → TypedDict

In [7]:
from pydantic import BaseModel, Field
from typing import Optional

class student(BaseModel):
    name: str = Field(..., description="The full name of the student")
    age: int = Field(..., description="The age of the student in years")
    grade: str = Field(..., description="The current grade level of the student")
    gpa: float = Field(gt=0.0,lt=5, description="The student's Grade Point Average")
    email: Optional[str] = Field(None, description="The student's email address")

s=student(name="John Doe",age=20,grade="Junior",gpa="4.8")
print(s)
#s1=student(name="Jane Smith",age="Twenty",grade="Senior",gpa=3.9) #passing the age in str format instead of int but this will raise error at runtime as pydantic validate the data at runtime
s2=student(name="Jane Smith",age=21,grade="Senior",gpa=3.9,email="abs@daasai.com")
print(s2)

name='John Doe' age=20 grade='Junior' gpa=4.8 email=None
name='Jane Smith' age=21 grade='Senior' gpa=3.9 email='abs@daasai.com'


In [8]:
student_json=s2.model_dump_json()
print(student_json)

{"name":"Jane Smith","age":21,"grade":"Senior","gpa":3.9,"email":"abs@daasai.com"}


In [9]:

from typing import TypedDict, Annotated, Optional, Literal
from pydantic import BaseModel, Field


# schema
class Review(BaseModel):

    key_themes: list[str] = Field(description="Write down all the key themes discussed in the review in a list")
    summary: str = Field(description="A brief summary of the review")
    sentiment: Literal["pos", "neg"] = Field(description="Return sentiment of the review either negative, positive or neutral")
    pros: Optional[list[str]] = Field(default=None, description="Write down all the pros inside a list")
    cons: Optional[list[str]] = Field(default=None, description="Write down all the cons inside a list")
    name: Optional[str] = Field(default=None, description="Write the name of the reviewer")
    

structured_model = model.with_structured_output(Review,model_return_type="pydantic")

result = structured_model.invoke("""I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I’m gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful
                                 
Review by Nitish Singh
""")

print(result)

ValueError: Received unsupported arguments {'model_return_type': 'pydantic'}

In [10]:
from typing import TypedDict, Annotated, Optional, Literal
from pydantic import BaseModel, Field


# schema
json_schema = {
  "title": "Review",
  "type": "object",
  "properties": {
    "key_themes": {
      "type": "array",
      "items": {
        "type": "string"
      },
      "description": "Write down all the key themes discussed in the review in a list"
    },
    "summary": {
      "type": "string",
      "description": "A brief summary of the review"
    },
    "sentiment": {
      "type": "string",
      "enum": ["pos", "neg"],
      "description": "Return sentiment of the review either negative, positive or neutral"
    },
    "pros": {
      "type": ["array", "null"],
      "items": {
        "type": "string"
      },
      "description": "Write down all the pros inside a list"
    },
    "cons": {
      "type": ["array", "null"],
      "items": {
        "type": "string"
      },
      "description": "Write down all the cons inside a list"
    },
    "name": {
      "type": ["string", "null"],
      "description": "Write the name of the reviewer"
    }
  },
  "required": ["key_themes", "summary", "sentiment"]
}


structured_model = model.with_structured_output(json_schema)

result = structured_model.invoke("""I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether 
I’m gaming, multitasking, or editing photos. 
The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.
The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. 
What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. 
Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.
However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—
why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful
                                 
Review by Nitish Singh
""")

print(result)

{'cons': ['Weight and size make it uncomfortable for one-handed use', 'Bloatware in One UI with unnecessary Samsung apps', 'High price tag of $1,300', 'Zoom quality degrades beyond 30x'], 'key_themes': ['High performance processor', 'Stunning camera and zoom', 'Long battery life and fast charging', 'S-Pen integration', 'Bloatware and extra apps', 'Premium price'], 'name': 'Nitish Singh', 'pros': ['Insanely powerful processor (great for gaming and productivity)', 'Stunning 200MP camera with incredible zoom capabilities', 'Long battery life with fast charging', 'S-Pen support is unique and useful'], 'sentiment': 'pos', 'summary': 'The Samsung Galaxy S24 Ultra impresses with its powerful Snapdragon 8 Gen 3 processor, exceptional 200MP camera with strong zoom, long battery life and fast charging, and useful S-Pen support. However, its large size makes one‑handed use awkward, it suffers from bloatware in One UI, and its hefty $1,300 price is a drawback.'}


# Output parsher:


In [ ]:
from langchain_core.prompts import PromptTemplate
#1st prompt -> detailed report o
template1= PromptTemplate(
    input_variables=["topic"],
    template="""
    write a detailed summary report on the country {country}
    """
)

#2nd prompt ->
template2= PromptTemplate(
    input_variables=["text"],
    template="""
    write a 5 line sumary on the folowing {text}
    """
)
prompt1=template1.invoke({'country':'artificial intelligence'})
result = model.invoke(prompt1)

prompt2=template2.invoke({'text':result.content})

result=model.invoke(prompt2)

print(result.content)

Artificial Intelligence (AI) is a multidisciplinary field that creates systems capable of performing tasks that once required human intelligence, from autonomous vehicles to medical diagnostics.  
Its history spans from Turing’s early theories, through expert systems and neural networks, to today’s foundation models such as GPT‑4 and Stable Diffusion.  
Core technologies include machine learning, deep learning, reinforcement learning, NLP, computer vision, generative models, symbolic AI, and hybrid neuro‑symbolic systems.  
These technologies power applications across healthcare, finance, transportation, and many other sectors, delivering faster diagnostics, risk assessment, automation, and decision‑support.  
Future research emphasizes democratization, ethical governance, explainability, and safe integration of AI into societal frameworks.


# Stroutputparser :
it use to extract llm response into str ouput using parser 
like above  

In [13]:
from langchain_core.output_parsers import StrOutputParser
parser =StrOutputParser()
chain =template1 | model | parser
rep =chain
result=rep.invoke({'topic':'machine learning'})
print(result)

# Detailed Summary Report  
**Machine Learning: Concepts, History, Applications, and Future Directions**  

---

## 1. Executive Summary  

Machine Learning (ML) is a subfield of artificial intelligence (AI) that equips computers with the ability to learn from data and make decisions or predictions without being explicitly programmed for each task. Over the past decades, advances in data availability, computational power, and algorithmic innovation have transformed ML from an academic curiosity into a cornerstone technology powering a wide array of industries—from healthcare and finance to autonomous vehicles and natural language processing. This report provides a comprehensive overview of the field, including its origins, core methodologies, key algorithms, notable applications, ongoing challenges, and emerging research directions.  

---

## 2. Historical Context  

| Era | Milestones & Key Figures | Technological Drivers |
|-----|-------------------------|-----------------------|
| 

In [14]:
rep.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
   +-----------------+     
   | ChatHuggingFace |     
   +-----------------+     
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  


In [15]:
from langchain_core.output_parsers import StrOutputParser

parser=StrOutputParser()

chain =template1 | model | parser | template2 | model | parser  

result1=chain.invoke({'topic':'artificial intelligence'})
print(result1)

- AI blends learning, reasoning, perception and language to automate, personalize and unlock insight across all industries.  
- Key milestones—from the 1956 Dartmouth conference to GPT‑4 in 2024—show a shift from rule‑based systems to data‑driven deep learning and transformers.  
- Core methods include supervised, unsupervised and reinforcement learning, along with symbolic, hybrid and multimodal models.  
- Real‑world deployments span autonomous vehicles, medical diagnostics, recommendation engines, and customer‑service chatbots.  
- Remaining hurdles involve data bias, explainability, safety, and ethical governance as AI scales to new domains.


In [16]:
chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
   +-----------------+     
   | ChatHuggingFace |     
   +-----------------+     
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *       

# JSONOutputparser :
theJsonoutputparser is parser force a model to generate the response into json format.

In [17]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.prompts import PromptTemplate
# 1st prompt  -> detailed report 

parser =JsonOutputParser()

template1 = PromptTemplate(
    input_variables=["topic"],
    template="""
    Give me 5 facts about {topic} \n {format_instruction}
    """,
    partial_variables={
        "format_instruction": parser.get_format_instructions()}
)
jsonchain =template1 | model | parser
result=jsonchain.invoke({'topic':'machine learning'})
print(result, type(result))
print(jsonchain.get_graph().print_ascii())

{'facts': ['Machine learning models learn patterns from data, enabling them to make predictions or decisions without explicit programming.', 'Supervised learning requires labeled data, while unsupervised learning discovers hidden structures in unlabeled data.', 'Deep learning, a subfield of machine learning, uses neural networks with many layers to model complex functions and achieve state-of-the-art performance in areas like image and speech recognition.', 'Overfitting occurs when a model captures noise instead of underlying patterns, leading to poor generalization on new data.', 'Transfer learning allows models trained on one task to be fine‑tuned for a related task, reducing the need for large labeled datasets.']} <class 'dict'>
      +-------------+      
      | PromptInput |      
      +-------------+      
             *             
             *             
             *             
    +----------------+     
    | PromptTemplate |     
    +----------------+     
      

# StructuredOutputParser :
StructuredOutputParser is an output parser in langchain that helps extract structured JSON data from LLM responses based omn predefined field schemas. 
- > It work by defining a list of fields (Response schema) that the model should returne, ensuring the output follows a structured format.

In [20]:
# from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
# import os
# from dotenv import load_dotenv  
# from langchain_core.prompts import PromptTemplate
# from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
# from langchain.output_parsers import StructuredOutputParser, ResponseSchema
# load_dotenv()

# #define themodel 
# llm =ChatHuggingFace(llm=HuggingFaceEndpoint(
#     repo_id="google/gemma-7b",
#     task="text-generation",
# ))

# schema =[
#     ResponseSchema(name="key_themes",description="Key themes discussed in the review"),
#     ResponseSchema(name="summary",description="A concise summary of the review"),
#     ResponseSchema(name="sentiment",description="The overall sentiment of the review, e.g., positive, negative, neutral"),
#     ResponseSchema(name="pros",description="List of pros mentioned in the review"),
#     ResponseSchema(name="cons",description="LIST OF CONS MENTIONS IN THE REVIEW"),
#     ResponseSchema(name="name",description="name of reviewer")
# ]

# parser =StructuredOutputParser.from_response_schemas(schema)

# template1 = PromptTemplate(
#     input_variables=["topic"],
#     template="""
#     Give me 5 facts about {topic} \n {format_instruction}
#     """,
#     partial_variables={
#         "format_instruction": parser.get_format_instructions()}
# )
# structured_chain =template1 | llm | parser
# result=structured_chain.invoke({'topic':'machine learning'})
# print(result)
# print(structured_chain.get_graph().print_ascii())


# Pydanticoutputparser :
PydanticOutputParser LangChain ka tool hai jo:
LLM ke free-text output ko
➡️ structured, validated Python object (Pydantic model) mein convert karta hai.

- Matlab:

    - LLM → jo text diya

    - Parser → usko JSON jaisa structure banake : pichle cell mein jsonoutputparser ab kaam nhi kr raha

    - Pydantic → type + validation check

    - Agar format galat → error. No bakwaas allowed.

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
import os
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
load_dotenv()

model =ChatHuggingFace(llm=HuggingFaceEndpoint(
    repo_id='google/gemma-2-2b-it',
    task='text-generation',
    huggingfacehub_api_token=os.getenv("HUGGINFACE_API_KEY")
))

class Facts(BaseModel):
    name_head :str= Field(..., description="The name of the Head of state of this country")
    population: int = Field(..., description="The population of the country")
    area: str = Field(..., description="The total area of the country in square kilometers")
    capital: str = Field(..., description="The capital city of the country")
    GDP:int= Field(gt=0, description="The current GDP of the country in USD")
    inflation_rate: float = Field(gt=1.0, description="The current inflation rate of the country in percentage")
    education_index: float = Field(gt=0.0, lt=1.0, description="The education index of the country")
    life_expectancy: float = Field(gt=0.0, description="The average life expectancy in the country")
    fact1: str = Field(..., description="The first fact about the country")
    fact2: str = Field(..., description="The second fact about the country")
    fact3: str = Field(..., description="The third fact about the country")
    fact4: str = Field(..., description="The fourth fact about the country")
    fact5: str = Field(..., description="The fifth fact about the country")

parser =PydanticOutputParser(pydantic_object=Facts)

template1 =PromptTemplate(
    template="""
    give me informations: name_of head leader, population, area, capital,GDP, capital, inflation_rate, education_index, life_expectancy, fact1, fact2, fact3, fact4, fact5
    about the country {country_name} \n {format_instructions}
    """,
    input_variables=["country_name"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()}
)

chain =template1 | model | parser
result=chain.invoke({'country_name':'India'})
print(result)

name_head='President Droupadi Murmu' population=1380000000 area='3287590' capital='New Delhi' GDP=346300000000000000 inflation_rate=6.7 education_index=0.529 life_expectancy=70.4 fact1="India is the world's second-most populous country." fact2='The Ganges River is a sacred river for Hindus.' fact3='India is home to many diverse religions and cultures.' fact4='The Taj Mahal is a UNESCO World Heritage site.' fact5='India has a rapidly developing economy.'


In [29]:
import json
with open('country_facts.json','w') as f:
    json.dump(result.model_dump(),f,indent=4)   